In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
from pathlib import Path

IMAGE_DIR = Path("/content/fotos")

In [ ]:
# 1. Instalación
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "torch", "pillow"])

# 2. Imports
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path
from collections import defaultdict
from transformers import Owlv2Processor, Owlv2ForObjectDetection

# 3. Atributos de vestimenta
text_labels = [[
    "historical costume", "baroque coat", "crinoline dress", "folk costume",
    "traditional costume", "military uniform", "naval uniform", "cassock",
    "nun habit", "tailcoat", "tuxedo", "morning coat", "three-piece suit",
    "evening gown", "veil", "mantilla", "fan", "shawl", "overcoat", "cloak",
    "top hat", "bow tie", "necktie", "gloves", "walking cane",
    "flamenco dress", "matador costume", "kimono", "turban", "laurel crown",
    "medieval costume", "renaissance dress", "sailor suit", "handbag"
]]

# 4. Cargar modelo (igual que en la doc oficial)
print("Cargando modelo...")
processor = Owlv2Processor.from_pretrained("google/owlv2-base-patch16-ensemble")
model     = Owlv2ForObjectDetection.from_pretrained("google/owlv2-base-patch16-ensemble")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Modelo listo en {device}")

# 5. Rutas
IMAGE_DIR  = Path("/content/fotos")
OUTPUT_DIR = Path("/content/resultados")
OUTPUT_DIR.mkdir(exist_ok=True)

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
image_paths = [p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in EXTENSIONS]
print(f"Imagenes encontradas: {len(image_paths)}")

# 6. Procesar cada imagen
all_results = []

for img_path in sorted(image_paths):
    print(f"\nProcesando: {img_path.name}")

    image = Image.open(img_path).convert("RGB")

    # -- Inferencia (código de la doc oficial) --
    inputs = processor(text=text_labels, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([(image.height, image.width)]).to(device)

    results = processor.post_process_grounded_object_detection(
        outputs=outputs,
        target_sizes=target_sizes,
        threshold=0.1,
        text_labels=text_labels
    )

    result = results[0]
    boxes      = result["boxes"]
    scores     = result["scores"]
    det_labels = result["text_labels"]

    # -- Mostrar detecciones (igual que en la doc oficial) --
    for box, score, label in zip(boxes, scores, det_labels):
        box = [round(i, 2) for i in box.tolist()]
        print(f"  Detected {label} with confidence {round(score.item(), 3)} at location {box}")

    # -- Dibujar bounding boxes --
    img_draw = image.copy()
    draw = ImageDraw.Draw(img_draw)
    for box, score, label in zip(boxes, scores, det_labels):
        xmin, ymin, xmax, ymax = [round(v, 1) for v in box.tolist()]
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=3)
        draw.text((xmin, max(0, ymin - 14)), f"{label} {score:.2f}", fill="red")

    img_draw.save(OUTPUT_DIR / img_path.name)

    # -- Mostrar en Colab --
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(image);     axes[0].set_title("Original");    axes[0].axis("off")
    axes[1].imshow(img_draw);  axes[1].set_title("Detecciones"); axes[1].axis("off")
    plt.suptitle(img_path.name)
    plt.tight_layout()
    plt.show()

    # -- Acumular para CSV (score máximo por atributo) --
    summary = defaultdict(float)
    for score, label in zip(scores, det_labels):
        summary[label] = max(summary[label], round(score.item(), 4))

    row = {"imagen": img_path.name}
    row.update({attr: round(summary.get(attr, 0.0), 4) for attr in text_labels[0]})
    all_results.append(row)

# 7. Exportar CSV
df = pd.DataFrame(all_results)
df.to_csv(OUTPUT_DIR / "resultados.csv", index=False)
print(f"\nCSV guardado en /content/resultados/resultados.csv")
df

In [ ]:
# ── Celda: Guardar en Drive + generar grafo_vestimenta.gexf ────────────
import networkx as nx
import os

DRIVE_OUTPUT = '/content/drive/MyDrive/TFM-Sara/output/vestimenta'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

G = nx.Graph()

# Nodos imagen
for row in all_results:
    fname = row['imagen'].strip()
    G.add_node(fname,
               label=fname,
               dimension='imagen',
               group=1,
               color='#4A90D9',
               size=25.0)

# Nodos vestimenta + aristas
for row in all_results:
    fname = row['imagen'].strip()
    for label in text_labels[0]:
        score = row.get(label, 0.0)
        if score > 0.0:
            if not G.has_node(label):
                G.add_node(label,
                           label=label,
                           dimension='vestimenta',
                           color='#9B59B6',
                           size=15.0)
            if G.has_edge(fname, label):
                G[fname][label]['weight'] = max(G[fname][label]['weight'], float(score))
            else:
                G.add_edge(fname, label,
                           relation='lleva_puesto',
                           weight=float(score),
                           dimension='vestimenta')

# Guardar GEXF en Drive
gexf_path = os.path.join(DRIVE_OUTPUT, 'grafo_vestimenta.gexf')
nx.write_gexf(G, gexf_path)
n_img  = sum(1 for _, d in G.nodes(data=True) if d.get('dimension') == 'imagen')
n_vest = sum(1 for _, d in G.nodes(data=True) if d.get('dimension') == 'vestimenta')
print(f'Grafo vestimenta: {G.number_of_nodes()} nodos '
      f'({n_img} imagen + {n_vest} vestimenta), '
      f'{G.number_of_edges()} aristas')
print(f'  -> {gexf_path}')

# Guardar tambien CSV en Drive
csv_drive = os.path.join(DRIVE_OUTPUT, 'resultados_vestimenta.csv')
df.to_csv(csv_drive, index=False)
print(f'CSV guardado: {csv_drive}')
